# Sesión 28 · Ecosistema y pipelines

Resuelve tareas reales con una línea y aprende a elegir modelos en el Hub.

**Objetivos:**
1. Usar `pipeline()` para varias tareas NLP.
2. Leer model cards y elegir el modelo adecuado.
3. Comparar dos modelos de sentimiento y ver el impacto del idioma.
4. Gestionar la caché y el token de autenticación.

> **Formato:** ~30 min explicación · ~75 min notebook · ~15 min reto.

In [ ]:
%pip install -q transformers torch pandas huggingface_hub

## 0 · Detección de dispositivo

En Colab puedes tener GPU (CUDA), en un Mac Apple Silicon tendrás MPS,
y en la mayoría de los casos solo CPU. Este patrón elige automáticamente.

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon (MPS)")
else:
    device = "cpu"
    print("CPU (sin aceleración GPU)")

print(f"\nUsando: {device}")

---
## 1 · Una pipeline de sentimiento

Al usar `pipeline()` sin especificar modelo, HF descarga el modelo por defecto
para esa tarea. Para reproducibilidad, fija siempre la versión con `revision=`.

In [ ]:
import pandas as pd
from transformers import pipeline

# revision= fija el commit exacto del modelo → resultados reproducibles
clasificador = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    revision="af0f99b"
)

reseña = """
I thought this was a wonderful way to spend a hot summer weekend,
sitting in the air-conditioned theater watching a light-hearted comedy.
The dialogue is witty and the characters are very likable.
"""

resultado = clasificador(reseña)
pd.DataFrame(resultado)

### Clasificar varios textos a la vez

In [ ]:
frases = [
    "I loved the product, it arrived ahead of schedule.",
    "Terrible quality, it broke on the second day.",
    "It was okay, nothing special but not bad either.",
]

resultados = clasificador(frases)
pd.DataFrame(resultados)

---
## 2 · Varias tareas con la misma interfaz

`pipeline()` acepta más de 20 tareas. La arquitectura interna cambia,
pero la API es siempre la misma.

In [ ]:
# NER: reconocimiento de entidades nombradas
ner = pipeline("ner", aggregation_strategy="simple")
entidades = ner("Apple abrió una nueva tienda en Valencia, España.")
pd.DataFrame(entidades)

In [ ]:
# Zero-shot: clasifica sin haber visto ejemplos de esas etiquetas
zero_shot = pipeline("zero-shot-classification")
resultado_zs = zero_shot(
    "Quiero devolver un producto que llegó roto",
    candidate_labels=["reclamacion", "compra", "consulta", "devolución"]
)
pd.DataFrame({"etiqueta": resultado_zs["labels"], "score": resultado_zs["scores"]})

In [ ]:
# Generación de texto
generador = pipeline(
    "text-generation",
    model="openai-community/gpt2"
)
salida = generador(
    "In this course, we will teach you how to",
    max_length=50,
    num_return_sequences=2,
    do_sample=True,
    temperature=0.9
)
for i, s in enumerate(salida, 1):
    print(f"\n--- Secuencia {i} ---")
    print(s["generated_text"])

---
## 3 · Elegir modelo: impacto del idioma

El modelo por defecto está entrenado en inglés. Para español necesitas uno
multilingüe o específico. La model card del Hub te dice el idioma soportado.

In [ ]:
modelo_ingles = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    revision="af0f99b"
)

modelo_multilingue = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

frases_esp = [
    "El pedido llegó roto, muy decepcionado.",
    "Me ha encantado el producto, funciona perfecto.",
    "Ni bien ni mal, simplemente normal.",
]

print(f"{'Frase':45s}  {'Inglés':18s}  {'Multilingüe'}")
print("-" * 85)
for frase in frases_esp:
    r_en = modelo_ingles(frase)[0]
    r_ml = modelo_multilingue(frase)[0]
    print(f"{frase[:45]:45s}  {r_en['label']:6s} {r_en['score']:.2f}  {r_ml['label']:6s} {r_ml['score']:.2f}")

---
## Práctica

1. Escribe **10 frases propias** en español (5 positivas, 5 negativas).
2. Clasifícalas con el modelo inglés y el multilingüe.
3. ¿Cuál acierta más? ¿Hay diferencias en el `score`?
4. Busca en el Hub un modelo de sentimiento **específico para español**
   y repite la comparación.

In [ ]:
# TODO: compara tres modelos sobre 10 frases propias en español
mis_frases = [
    "...",  # positiva
    "...",  # negativa
]

---
## Reto · Elegir el mejor modelo de NER en español

1. Ve a https://huggingface.co/models y filtra por **tarea = Token Classification** e **idioma = es**.
2. Elige el modelo que te parezca más adecuado y justifica la elección con su model card:
   - ¿Qué tarea resuelve exactamente?
   - ¿Qué métricas reporta?
   - ¿Qué licencia tiene?
3. Pruébalo en el notebook y compara con el modelo NER por defecto.

In [ ]:
# TODO: probar un modelo de NER en español y justificar la elección
# ner_es = pipeline("ner", model="...", aggregation_strategy="simple")
# print(ner_es("El Ayuntamiento de Valencia aprobó el presupuesto para 2027."))

---
## 4 · Cuenta, CLI y token de autenticación

Para descargar modelos privados o publicar en el Hub necesitas autenticarte.
Desde el terminal: `hf auth login` (o `huggingface-cli login`).

El token puede ser de **lectura** (solo descargar) o **escritura** (publicar).
Nunca lo compartas ni lo subas a un repositorio público.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
# TODO: ejecuta 'hf auth login' en el terminal y luego descomenta:
# print(api.whoami())

# Descargar un archivo concreto de un modelo (sin descargar el modelo entero)
ruta = hf_hub_download(
    repo_id="google/pegasus-xsum",
    filename="config.json"
)
print("Config guardada en:", ruta)

---
## Autoevaluación

| Pregunta | Respuesta |
|---|---|
| ¿Para qué sirve el parámetro `revision=` en `pipeline()`? | |
| ¿Qué devuelve `pipeline()` cuando le pasas una lista de textos? | |
| ¿Por qué el modelo inglés puede fallar con texto en español? | |
| ¿Cuál es la diferencia entre un token de lectura y de escritura? | |
| ¿Qué información obligatoria debe tener una model card? | |